# Treatment Outcomes in r/covidlonghaulers: LDN, Peptides, and POTS

**Research questions:**
1. Which treatments have the highest positive outcome rates among the most-discussed drugs?
2. Do patients with POTS respond differently to Low Dose Naltrexone (LDN) than patients without POTS?
3. Among users who tried multiple treatments, which pairings show different outcomes?

**Data:** 1-month snapshot (1,081 posts, ~17,000 posts+comments) from r/covidlonghaulers.  
**Methods:** Binomial test, Mann-Whitney U, Wilcoxon signed-rank (paired), Kruskal-Wallis.

## 1. Setup

In [ ]:
import sqlite3
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest, mannwhitneyu

DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "patientpunk.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "patientpunk.db"
conn = sqlite3.connect(DB_PATH)

SENTIMENT_SCORE = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0, "weak": 0.25}

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

## 2. Data Overview

In [ ]:
for table in ["users", "posts", "treatment", "treatment_reports", "user_profiles", "conditions"]:
    count = pd.read_sql(f"SELECT COUNT(*) as n FROM {table}", conn).iloc[0, 0]
    print(f"  {table}: {count:,}")

print("\nSentiment distribution:")
print(pd.read_sql("SELECT sentiment, COUNT(*) as n FROM treatment_reports GROUP BY sentiment ORDER BY n DESC", conn).to_string(index=False))

## 3. Question 1: Which treatments have the best outcomes?

We aggregate to the **user level** (one data point per user per drug) to ensure independence, then run a binomial test asking whether each drug's positive rate differs from 50% (chance).

In [ ]:
# Build user-level treatment outcome table
df = pd.read_sql("""
    SELECT tr.user_id, t.canonical_name AS drug, tr.sentiment
    FROM treatment_reports tr
    JOIN treatment t ON t.id = tr.drug_id
""", conn)
df["score"] = df["sentiment"].map(SENTIMENT_SCORE)

# Aggregate to user level
user_drug = df.groupby(["drug", "user_id"]).agg(
    avg_score=("score", "mean"),
    n_reports=("score", "count")
).reset_index()
user_drug["outcome"] = user_drug["avg_score"].apply(
    lambda x: "positive" if x > 0.7 else ("negative" if x < -0.3 else "mixed/neutral")
)

# Drugs with 5+ user reports
drug_counts = user_drug.groupby("drug")["user_id"].count().reset_index()
drug_counts.columns = ["drug", "n_users"]
top_drugs = drug_counts[drug_counts["n_users"] >= 5].sort_values("n_users", ascending=False)

# Binomial test for each
results = []
for _, row in top_drugs.iterrows():
    drug = row["drug"]
    users = user_drug[user_drug["drug"] == drug]
    n = len(users)
    n_pos = (users["outcome"] == "positive").sum()
    n_neg = (users["outcome"] == "negative").sum()
    test = binomtest(n_pos, n, p=0.5)
    results.append({
        "Drug": drug, "Users": n,
        "Positive": n_pos, "Negative": n_neg,
        "% Positive": round(n_pos/n*100, 1),
        "Avg Score": round(users["avg_score"].mean(), 2),
        "p-value": round(test.pvalue, 4),
        "Sig": "*" if test.pvalue < 0.05 else ""
    })

results_df = pd.DataFrame(results)
print(f"Treatments with 5+ user reports ({len(results_df)} drugs):")
print("(* = positive rate significantly different from 50%, p < 0.05)\n")
display(results_df)

In [ ]:
# Visualization: outcome breakdown for top drugs (5+ user reports only)
plot_drugs = results_df.head(15)
n_drugs = len(plot_drugs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(5, n_drugs * 0.5)))

drug_order = plot_drugs.sort_values("% Positive")["Drug"]
plot_data = user_drug[user_drug["drug"].isin(drug_order)]

# Left: stacked percentage bar
outcome_pct = plot_data.groupby(["drug", "outcome"]).size().unstack(fill_value=0)
outcome_pct = outcome_pct.div(outcome_pct.sum(axis=1), axis=0) * 100
outcome_pct = outcome_pct.reindex(drug_order)

colors = {"positive": "#2ecc71", "mixed/neutral": "#95a5a6", "negative": "#e74c3c"}
outcome_pct.plot(kind="barh", stacked=True, ax=ax1,
                  color=[colors.get(c, "gray") for c in outcome_pct.columns], edgecolor="white")

for i, drug in enumerate(drug_order):
    n = int(plot_drugs[plot_drugs["Drug"] == drug]["Users"].values[0])
    ax1.text(102, i, f"n={n}", va="center", fontsize=9, color="gray")

ax1.set_xlabel("Percentage of users")
ax1.set_title("Treatment Outcomes (% positive / mixed / negative)")
ax1.legend(title="Outcome", loc="lower right", fontsize=8)
ax1.set_xlim(0, 115)

# Right: box plot of sentiment scores
sns.boxplot(data=plot_data, y="drug", x="avg_score", ax=ax2, orient="h",
            order=drug_order, palette="coolwarm", fliersize=3)
ax2.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax2.set_xlabel("Average sentiment score")
ax2.set_title("Sentiment Distribution")
ax2.set_ylabel("")

plt.suptitle(f"Top {n_drugs} Treatments with 5+ User Reports (1-month sample)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Question 2: Do POTS patients respond differently to LDN?

We split LDN users into two groups — those with a POTS condition and those without — and compare their sentiment scores using a Mann-Whitney U test.

In [ ]:
# Get LDN users
ldn_users = user_drug[user_drug["drug"] == "low dose naltrexone"][["user_id", "avg_score", "outcome"]].copy()

# Get POTS users
pots_users = set(pd.read_sql("""
    SELECT DISTINCT user_id FROM conditions
    WHERE LOWER(condition_name) LIKE '%pots%'
""", conn)["user_id"])

ldn_users["has_pots"] = ldn_users["user_id"].isin(pots_users)

pots_group = ldn_users[ldn_users["has_pots"]]["avg_score"]
no_pots_group = ldn_users[~ldn_users["has_pots"]]["avg_score"]

print(f"LDN users with POTS: {len(pots_group)}")
print(f"LDN users without POTS: {len(no_pots_group)}")

if len(pots_group) >= 3 and len(no_pots_group) >= 3:
    stat, p = mannwhitneyu(pots_group, no_pots_group, alternative="two-sided")
    print(f"\nMann-Whitney U test:")
    print(f"  U statistic: {stat:.1f}")
    print(f"  p-value: {p:.4f}")
    print(f"  Significant (p < 0.05): {'Yes' if p < 0.05 else 'No'}")
    print(f"\n  POTS group mean: {pots_group.mean():.2f}")
    print(f"  Non-POTS group mean: {no_pots_group.mean():.2f}")
else:
    print("\nInsufficient data for statistical comparison (need 3+ in each group)")

In [ ]:
# Visualization
if len(pots_group) >= 1 and len(no_pots_group) >= 1:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Box plot
    ldn_users["Group"] = ldn_users["has_pots"].map({True: f"POTS (n={len(pots_group)})", False: f"No POTS (n={len(no_pots_group)})"})
    sns.boxplot(data=ldn_users, x="Group", y="avg_score", ax=axes[0], palette=["#e74c3c", "#3498db"])
    axes[0].set_ylabel("Average sentiment score")
    axes[0].set_title("LDN Outcomes: POTS vs Non-POTS")
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    
    # Outcome distribution
    outcome_by_pots = ldn_users.groupby(["Group", "outcome"]).size().unstack(fill_value=0)
    outcome_by_pots_pct = outcome_by_pots.div(outcome_by_pots.sum(axis=1), axis=0) * 100
    outcome_by_pots_pct.plot(kind="bar", stacked=True, ax=axes[1],
                              color=[colors.get(c, "gray") for c in outcome_by_pots_pct.columns])
    axes[1].set_ylabel("Percentage")
    axes[1].set_title("Outcome Distribution: POTS vs Non-POTS")
    axes[1].legend(title="Outcome")
    axes[1].tick_params(axis='x', rotation=0)
    
    plt.tight_layout()
    plt.show()

## 5. Question 3: Paired comparison — users who tried multiple treatments

For users who tried two or more of the top treatments, we can do a within-subject comparison. This eliminates confounders (severity, demographics) because we're comparing the same person's response to different drugs.

In [ ]:
# Find users who tried multiple top drugs
top_drug_names = results_df.head(10)["Drug"].tolist()
multi_users = user_drug[user_drug["drug"].isin(top_drug_names)].copy()
users_with_multiple = multi_users.groupby("user_id")["drug"].nunique()
users_with_multiple = users_with_multiple[users_with_multiple >= 2].index

print(f"Users who tried 2+ of the top {len(top_drug_names)} drugs: {len(users_with_multiple)}")

# Find the most common drug pairs
from itertools import combinations
pair_counts = {}
for uid in users_with_multiple:
    drugs = sorted(multi_users[multi_users["user_id"] == uid]["drug"].unique())
    for a, b in combinations(drugs, 2):
        pair = (a, b)
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

print("\nMost common drug pairs (users who tried both):")
for (a, b), n in sorted(pair_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {a} + {b}: {n} users")

In [ ]:
# Paired comparison for the most common pair
from scipy.stats import wilcoxon

if pair_counts:
    best_pair = max(pair_counts, key=pair_counts.get)
    drug_a, drug_b = best_pair
    n_shared = pair_counts[best_pair]
    
    if n_shared >= 5:
        # Get paired scores
        shared_users = set(users_with_multiple)
        a_scores = user_drug[(user_drug["drug"] == drug_a) & (user_drug["user_id"].isin(shared_users))].set_index("user_id")["avg_score"]
        b_scores = user_drug[(user_drug["drug"] == drug_b) & (user_drug["user_id"].isin(shared_users))].set_index("user_id")["avg_score"]
        paired = pd.DataFrame({"drug_a": a_scores, "drug_b": b_scores}).dropna()
        
        if len(paired) >= 5:
            diffs = paired["drug_a"] - paired["drug_b"]
            non_zero = diffs[diffs != 0]
            
            print(f"Paired comparison: {drug_a} vs {drug_b} ({len(paired)} shared users)")
            print(f"  Mean score {drug_a}: {paired['drug_a'].mean():.2f}")
            print(f"  Mean score {drug_b}: {paired['drug_b'].mean():.2f}")
            print(f"  Mean difference: {diffs.mean():.2f}")
            
            if len(non_zero) >= 5:
                stat, p = wilcoxon(non_zero)
                print(f"\n  Wilcoxon signed-rank test:")
                print(f"    W statistic: {stat:.1f}")
                print(f"    p-value: {p:.4f}")
                print(f"    Significant: {'Yes' if p < 0.05 else 'No'}")
                direction = f"{drug_a} better" if diffs.mean() > 0 else f"{drug_b} better" if diffs.mean() < 0 else "no difference"
                print(f"    Direction: {direction}")
            else:
                print(f"\n  Too few non-zero differences ({len(non_zero)}) for Wilcoxon test")
        else:
            print(f"Only {len(paired)} paired observations — need 5+ for Wilcoxon")
    else:
        print(f"Most common pair ({drug_a} + {drug_b}) has only {n_shared} shared users — need 5+")
else:
    print("No drug pairs found")

## 6. Multi-drug comparison (Kruskal-Wallis)

Are there overall differences in sentiment across the top treatments? Kruskal-Wallis tests whether at least one group differs.

In [ ]:
from scipy.stats import kruskal

# Get top 5 drugs with enough data
kw_drugs = results_df[results_df["Users"] >= 5].head(5)["Drug"].tolist()
kw_groups = [user_drug[user_drug["drug"] == d]["avg_score"].values for d in kw_drugs]

if len(kw_groups) >= 3:
    h_stat, kw_p = kruskal(*kw_groups)
    print(f"Kruskal-Wallis test across {len(kw_drugs)} drugs:")
    print(f"  Drugs: {', '.join(kw_drugs)}")
    print(f"  H statistic: {h_stat:.2f}")
    print(f"  p-value: {kw_p:.4f}")
    print(f"  Significant: {'Yes — at least one drug differs' if kw_p < 0.05 else 'No significant difference'}")

    # Box plot
    fig, ax = plt.subplots(figsize=(10, 5))
    kw_data = user_drug[user_drug["drug"].isin(kw_drugs)]
    sns.boxplot(data=kw_data, x="drug", y="avg_score", ax=ax, palette="Set2")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel("Treatment")
    ax.set_ylabel("User-level average sentiment")
    ax.set_title(f"Sentiment Distribution: Top {len(kw_drugs)} Treatments (Kruskal-Wallis p={kw_p:.3f})")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Fewer than 3 drugs with sufficient data for Kruskal-Wallis")

## 7. Condition prevalence

In [ ]:
conditions = pd.read_sql("""
    SELECT condition_name, COUNT(DISTINCT user_id) as n_users
    FROM conditions
    GROUP BY condition_name
    ORDER BY n_users DESC
    LIMIT 15
""", conn)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=conditions, x="n_users", y="condition_name", ax=ax, color="#3498db")
ax.set_xlabel("Number of users")
ax.set_ylabel("")
ax.set_title("Top 15 Conditions in r/covidlonghaulers (1-month sample)")
plt.tight_layout()
plt.show()

## 8. Summary

### Key findings

**Q1: Which treatments have the best outcomes?**
- 526 treatment reports across 177 canonicalized drugs from a 1-month r/covidlonghaulers snapshot
- Top drugs by volume: supplements (39), LDN (22), peptides (19), tirzepatide (19), taurine (18)
- Binomial tests identify which drugs have positive rates significantly above/below chance

**Q2: POTS and LDN?**
- Mann-Whitney U compares LDN sentiment between POTS and non-POTS users
- This tests whether the comorbidity affects treatment response

**Q3: Paired comparisons?**
- Users who tried multiple treatments provide within-subject comparisons
- Wilcoxon signed-rank eliminates between-subject confounders

### Caveats

- **Sample sizes remain modest** — most drugs have 5-22 user reports. Statistical power is limited.
- **Canonicalization issues** — generic terms like "supplement" and "medication" are over-counted. Some conditions may be tagged as treatments.
- **Sentiment is self-reported and self-selected** — users who post about treatments may skew toward strong opinions.
- **1-month window** — seasonal or trending effects may bias which treatments are discussed.

### Next steps

- Run the LLM backfill pipeline to improve demographic coverage (currently regex-only at 29.6% fill rate)
- Download 3-6 months of data for more robust per-drug sample sizes
- Improve canonicalization to separate conditions from treatments
- Run logistic regression to identify predictors of positive outcomes controlling for demographics

---

*Based on self-selected Reddit posts. Users who never posted about a treatment are not represented. Results reflect reporting patterns, not population-level treatment effects.*

In [ ]:
conn.close()